In [25]:
import re
import requests
from bs4 import BeautifulSoup, Tag
from typing import Dict, List

In [2]:
base_url = "http://www.vestibular.ita.br"
estatisticas_url = f"{base_url}/estatisticas.htm"

In [ ]:
# Step 1: Fetch "Estatísticas" page
response = requests.get(estatisticas_url)
if response.status_code == 200:
    print("Successfully fetched Estatísticas page!")
    soup = BeautifulSoup(response.text, 'html.parser')
else:
    print("Failed to fetch Estatísticas page:", response.status_code)

Successfully fetched Estatísticas page!


In [12]:
# Step 2: Extract all links to past vestibular statistics
vestibular_links = soup.find_all('a', href=True)

In [13]:
vestibular_links[0]

<a href="#dados2025"><strong><img border="0" height="25" src="img/visto.gif" width="23"/>Vestibular 
              de 2025</strong></a>

In [19]:

for link in vestibular_links[:1]:
    text = link.text.strip()
    href = link['href']
    print(f"{text} -> {href}")

# Step 3: Extract any tables or key data
tables = soup.find_all('table')
for idx, table in enumerate(tables[:1]):
    print(f"\n--- Table {idx + 1} ---")
    rows = table.find_all('tr')
    for row in rows:
        cols = row.find_all(['td', 'th'])
        with open('statistics.txt', 'a') as f:
            f.write(str([col.text.strip() for col in cols]) + '\n')

Vestibular 
              de 2025 -> #dados2025

--- Table 1 ---


In [77]:
import re
import requests
from bs4 import BeautifulSoup, Tag
from typing import Dict, List, Tuple

def fetch_estatisticas_sections(url: str) -> Dict[str, BeautifulSoup]:
    """
    Fetches the HTML from 'estatisticas.htm', splits it by year anchor sections,
    and returns a dictionary: { '2025': soup_for_2025_section, '2024': soup_for_2024_section, ... }.
    """
    response = requests.get(url)
    response.raise_for_status()
    
    main_soup = BeautifulSoup(response.text, 'html.parser')
    
    # Anchors have IDs like 'dados2025', 'dados2024', etc.
    # We want them in the order they appear, so let's find them in the DOM order.
    all_anchors = main_soup.find_all(id=re.compile(r'^dados20\d{2}$'))
    
    if not all_anchors:
        print("No anchors of the form #dados20XX found.")
        return {}
    
    sections: Dict[str, BeautifulSoup] = {}
    
    # Iterate over each anchor, gather all content until the next anchor
    for i, anchor in enumerate(all_anchors):
        # Extract the year (e.g. from 'dados2025' -> '2025')
        year_match = re.search(r'(\d{4})$', anchor.get('id', ''))
        if not year_match:
            continue
        year = year_match.group(1)

        # "Stop node": the next anchor in the list, or None if last anchor
        next_anchor = all_anchors[i + 1] if i + 1 < len(all_anchors) else None

        # Collect everything from this anchor up to (but not including) the next anchor
        content_nodes = []
        node = anchor
        while node and node != next_anchor:
            content_nodes.append(node)
            node = node.next_sibling

        # Combine these nodes into a single HTML string, parse again
        combined_html = ''.join(str(n) for n in content_nodes)
        section_soup = BeautifulSoup(combined_html, 'html.parser')
        
        sections[year] = section_soup
    
    return sections

# def get_table_title(table: Tag) -> str:
#     """
#     Heuristically look upward in the HTML for a heading-like element that names the table.
#     Commonly, a <strong> or <font><strong> line just above the table indicates its title.

#     Returns the text if found, else "No Title Found".
#     """
#     # Move backward through siblings until we find a strong or some text that looks like a heading.
#     label_candidate = table.previous_sibling
#     while label_candidate:
#         # If it's just a newline or blank string, skip
#         if isinstance(label_candidate, str) and not label_candidate.strip():
#             label_candidate = label_candidate.previous_sibling
#             continue
        
#         # If it's a Tag, see if it contains a <strong> or bold text
#         if isinstance(label_candidate, Tag):
#             # Look for a <strong> element
#             strong_el = label_candidate.find('strong')
#             if strong_el:
#                 # Use the text from that <strong>
#                 text = strong_el.get_text(strip=True)
#                 if text:
#                     return text
#         label_candidate = label_candidate.previous_sibling
    
#     return "No Title Found"

def get_table_title(table: Tag) -> str:
    """
    Revised approach: move up the DOM to find any tag whose text begins with "<number> -".
    This skips unrelated <strong> elements (like "Prova"), ensuring we catch headings
    like "2 - Especialidade escolhida em primeira opção".
    """
    import re
    
    # Use `find_previous()` to look upward in the DOM for a matching element
    heading = table.find_previous(
        lambda t: (
            isinstance(t, Tag)
            and t.get_text(strip=True)
            # Matches lines like "2 - Especialidade..."
            and re.match(r'^\d+\s*-\s+', t.get_text(strip=True))
        )
    )
    return heading.get_text(strip=True) if heading else "No Title Found"



def parse_year_tables(year_soup: BeautifulSoup) -> List[Tuple[str, List[List[str]]]]:
    """
    Parse the tables from a single year’s section.
    Returns a list of (table_title, table_data),
    where table_data is a list of rows, and each row is a list of cell texts.
    """
    result = []
    tables = year_soup.find_all('table')
    
    for tbl in tables:
        # 1) Find a name/heading for this table
        table_title = get_table_title(tbl)
        
        # 2) Extract rows from the table
        rows_data = []
        for row in tbl.find_all('tr'):
            cells = [cell.get_text(strip=True) for cell in row.find_all(['td', 'th'])]
            rows_data.append(cells)
        
        result.append((table_title, rows_data))
    
    return result

def clean_title(raw_title: str) -> str:
    match = re.match(r'^\d+\s*-\s+(.*)', raw_title, flags=re.DOTALL)
    remaining_text = match.group(1) or raw_title
    cleaned = re.sub(r'\s+', ' ', remaining_text).strip()
    return cleaned

In [63]:
year_sections = fetch_estatisticas_sections(estatisticas_url)
year_sections = {k: v for k, v in year_sections.items() if k == '2025'}

In [81]:
# Vestibular data will store a list of tables for each year
vestibular_data = {} 

# 2) For each year, parse the tables
for year, soup_section in year_sections.items():
    print(f"==== Vestibular de {year} ====")
    
    # Example: parse the tables
    table_info = parse_year_tables(soup_section)
    for (title, values) in table_info:
        if year not in vestibular_data:
            vestibular_data[year] = [{"title": clean_title(title), "values": values}]
        else:
            vestibular_data[year].append({"title": clean_title(title), "values": values})

    #     print(f"  >> Table Title: {title}")
    #     for row in values:
    #         print("    ", row)
    
    # print("\n")

==== Vestibular de 2025 ====


In [79]:
[t['title'] for t in vestibular_data['2025']]

['Número de candidatos inscritos',
 'Especialidade escolhida em primeira opção',
 'Procedência escolar dos candidatos inscritos',
 'Abstenção às provas',
 'Médias dos candidatos convocados',
 'Nota de corte (todas as chamadas)',
 'Candidatos Convocados (Todos)',
 'Bancas Examinadoras e número de candidatos inscritos em cada uma delas']

In [80]:
vestibular_data['2025'][0]

{'title': 'Número de candidatos inscritos',
 'values': [['Tipo', 'Homens', 'Mulheres', 'Total', ''],
  ['Efetivos', '6.224', '1.733', '7.957', '81,4%'],
  ['Treineiros', '1.374', '450', '1.824', '18,6%'],
  ['Total', '7.598', '2.183', '9.781', '']]}